# Hybrid MBTI Report

Notebook này dùng để theo dõi pipeline hybrid tree-based sau khi gộp dữ liệu cũ và audio mới.

Workflow gọn:
1. `crawl-metadata`
2. `crawl-audio`
3. `extract-cnn`
4. `train-hybrid`

Entry script thống nhất: `scripts/project_workflow.py`.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path(r"D:/project")
PYTHON_EXE = PROJECT_ROOT / "mbti_env" / "Scripts" / "python.exe"
WORKFLOW_SCRIPT = PROJECT_ROOT / "scripts" / "project_workflow.py"
HYBRID_METRICS_PATH = PROJECT_ROOT / "outputs" / "hybrid_tree_metrics.json"
AUDIO_FEATURE_CACHE = PROJECT_ROOT / "data" / "audio_tabular_features.csv"
TRAIN_MANIFEST_PATH = PROJECT_ROOT / "data" / "train_manifest.json"

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## Workflow Shortcuts

In [ ]:
def run_workflow(*args):
    cmd = [str(PYTHON_EXE), str(WORKFLOW_SCRIPT), *args]
    print(" ".join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)


In [ ]:
# Refresh hybrid end-to-end từ audio_files hiện có
# run_workflow("refresh-hybrid")

# Chỉ train lại hybrid tree baseline nếu feature cache đã có
# run_workflow("train-hybrid")


## Dataset Snapshot

In [ ]:
snapshot = {
    "train_manifest_exists": TRAIN_MANIFEST_PATH.exists(),
    "audio_tabular_feature_cache_exists": AUDIO_FEATURE_CACHE.exists(),
    "hybrid_metrics_exists": HYBRID_METRICS_PATH.exists(),
}
snapshot


In [ ]:
if TRAIN_MANIFEST_PATH.exists():
    manifest = json.load(open(TRAIN_MANIFEST_PATH, encoding="utf-8"))
    print(f"train_manifest samples: {len(manifest)}")
else:
    print("train_manifest.json is missing")

if AUDIO_FEATURE_CACHE.exists():
    feature_cache = pd.read_csv(AUDIO_FEATURE_CACHE)
    print(f"audio_tabular_features rows: {len(feature_cache)}")
    display(feature_cache.head())
else:
    print("audio_tabular_features.csv is missing")


## Hybrid Metrics

In [ ]:
if not HYBRID_METRICS_PATH.exists():
    raise FileNotFoundError(HYBRID_METRICS_PATH)

metrics = json.load(open(HYBRID_METRICS_PATH, encoding="utf-8"))
metrics


In [ ]:
summary_df = pd.DataFrame([
    {
        "dimension": dim,
        **payload,
    }
    for dim, payload in metrics["dimensions"].items()
])
summary_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=summary_df, x="dimension", y="test_accuracy", ax=axes[0], color="#2a9d8f")
axes[0].set_title("Hybrid Test Accuracy")
axes[0].set_ylim(0, 1)

sns.barplot(data=summary_df, x="dimension", y="test_f1", ax=axes[1], color="#e76f51")
axes[1].set_title("Hybrid Test F1")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()


## Quick Interpretation

In [ ]:
overall_accuracy = metrics["overall_test_accuracy"]
overall_f1 = metrics["overall_test_f1"]
print(f"overall_test_accuracy: {overall_accuracy:.4f}")
print(f"overall_test_f1: {overall_f1:.4f}")

best_dim = summary_df.sort_values("test_accuracy", ascending=False).iloc[0]
worst_dim = summary_df.sort_values("test_accuracy", ascending=True).iloc[0]
print(f"best dimension: {best_dim['dimension']} ({best_dim['test_accuracy']:.4f})")
print(f"worst dimension: {worst_dim['dimension']} ({worst_dim['test_accuracy']:.4f})")
